# 03 — Sobreajuste, regularización y validación

## Motivación

En el ejercicio de la sesión pasada viste un modelo que **se queda corto**: una recta
intentando describir una parábola. La solución parece obvia — usar un modelo más
flexible. Pero la flexibilidad tiene un costo: un modelo suficientemente flexible
puede ajustar *perfectamente* los datos de entrenamiento... **memorizando el ruido**.
Ese modelo obtiene pérdida cero en train y generaliza mal con datos nuevos.

Este fenómeno se llama **sobreajuste** (*overfitting*) y es el problema central del
machine learning. La sesión de hoy construye las tres herramientas para manejarlo:

1. **Diagnóstico** — comparar el error de train contra el de test
2. **Tratamiento** — la regularización
3. **Protocolo** — la validación cruzada para elegir hiperparámetros con honestidad

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_DIR = Path("../../datos")

rng = np.random.default_rng(seed=42)

## El experimento: ajustar polinomios a una ley conocida

Otra vez el experimento controlado. La ley verdadera será

$$y = \sin(2\pi x) + \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, 0.3^2)$$

con apenas **25 puntos** de entrenamiento. Ajustaremos polinomios de grado creciente:

$$f(x) = w_0 + w_1 x + w_2 x^2 + \cdots + w_g x^g$$

Nota: esto **sigue siendo regresión lineal** — el modelo es lineal *en los parámetros*
$w_j$. Solo cambiamos las características: en lugar de $x$, usamos $(x, x^2, \ldots, x^g)$.
En scikit-learn eso es `PolynomialFeatures` seguido de `LinearRegression`,
encadenados en un `Pipeline`.

In [ ]:
def true_function(x):
    """Ley que genera los datos (en la práctica nunca la conocemos)."""
    return np.sin(2 * np.pi * x)


noise_std = 0.3
n_train, n_test = 25, 300

x_train = rng.uniform(0, 1, n_train)
y_train = true_function(x_train) + rng.normal(0, noise_std, n_train)

x_test = rng.uniform(0, 1, n_test)
y_test = true_function(x_test) + rng.normal(0, noise_std, n_test)

x_grid = np.linspace(0, 1, 300)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(x_train, y_train, s=30, alpha=0.8, label="train (25 puntos)")
ax.plot(x_grid, true_function(x_grid), linewidth=2, color="C2", label="ley verdadera")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Pocos datos, ruido real: el escenario típico")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures


def fit_polynomial(degree, x, y):
    """Ajusta un polinomio del grado dado vía regresión lineal sobre potencias de x."""
    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    model.fit(x.reshape(-1, 1), y)
    return model


degrees_to_show = [1, 3, 15]

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, degree in zip(axes, degrees_to_show):
    model = fit_polynomial(degree, x_train, y_train)
    ax.scatter(x_train, y_train, s=25, alpha=0.7)
    ax.plot(x_grid, true_function(x_grid), linewidth=1.5, color="C2", alpha=0.7)
    ax.plot(x_grid, model.predict(x_grid.reshape(-1, 1)), linewidth=2, color="C1")
    ax.set_ylim(-1.8, 1.8)
    ax.set_xlabel("x")
    ax.set_title(f"grado {degree}")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("y")
fig.suptitle("Subajuste (grado 1) — ajuste razonable (3) — sobreajuste (15)", y=1.02)
fig.tight_layout()
plt.show()

El panel resume el fenómeno completo:

- **Grado 1** (recta): demasiado rígido — ni siquiera describe el train. *Subajuste.*
- **Grado 3**: captura la forma de la ley verdadera y suaviza el ruido.
- **Grado 15**: pasa casi exactamente por cada punto de train — **incluido el ruido** —
  y entre puntos oscila con amplitud creciente. *Sobreajuste.*

El detalle crucial: el modelo de grado 15 tiene el **menor** error de entrenamiento de
los tres. Si solo miráramos train, lo elegiríamos. Cuantifiquemos con la curva de
validación: error de train y de test como función del grado.

In [ ]:
from sklearn.metrics import root_mean_squared_error

degrees = range(0, 16)
rmse_train, rmse_test = [], []

for degree in degrees:
    model = fit_polynomial(degree, x_train, y_train)
    pred_train = model.predict(x_train.reshape(-1, 1))
    pred_test = model.predict(x_test.reshape(-1, 1))
    rmse_train.append(root_mean_squared_error(y_train, pred_train))
    rmse_test.append(root_mean_squared_error(y_test, pred_test))

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(degrees, rmse_train, linewidth=2, marker="o", markersize=5, label="train")
ax.plot(degrees, rmse_test, linewidth=2, marker="o", markersize=5, label="test")
ax.axhline(noise_std, color="C2", linewidth=1.5, linestyle="--",
           label="ruido irreducible (σ)")
ax.set_yscale("log")
ax.set_xlabel("grado del polinomio (complejidad del modelo)")
ax.set_ylabel("RMSE")
ax.set_title("La curva de validación: train baja siempre, test tiene un mínimo")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## Teoría: la descomposición sesgo–varianza

La curva anterior obedece a una ley general, no a una particularidad de este dataset.
Para el error cuadrático esperado de un modelo $\hat{f}$ entrenado sobre datasets
aleatorios, en un punto $x$:

$$\mathbb{E}\left[ (y - \hat{f}(x))^2 \right]
= \underbrace{\left( f(x) - \mathbb{E}[\hat{f}(x)] \right)^2}_{\text{sesgo}^2}
+ \underbrace{\mathbb{E}\left[ \left( \hat{f}(x) - \mathbb{E}[\hat{f}(x)] \right)^2 \right]}_{\text{varianza}}
+ \underbrace{\sigma^2}_{\text{ruido}}$$

(la derivación completa está en el **Apéndice A**). Los tres términos:

- **Sesgo**: error sistemático por rigidez del modelo. La recta tiene sesgo alto:
  aunque tuviéramos infinitos datos, jamás describiría el seno.
- **Varianza**: sensibilidad al dataset particular. El polinomio de grado 15 tiene
  varianza altísima — con otros 25 puntos del mismo proceso, daría una curva totalmente
  distinta (cada oscilación persigue puntos de ruido individuales).
- **Ruido irreducible** $\sigma^2$: la cota inferior del error que ningún modelo puede
  superar — la curva de test se aproxima a la línea punteada $\sigma$ pero no la cruza.

Aumentar la complejidad reduce el sesgo y aumenta la varianza: por eso el error
de test tiene forma de U. Elegir un modelo es elegir un punto en ese equilibrio.

## Regularización: penalizar la complejidad

¿Y si en lugar de restringir el *grado* restringimos los *pesos*? El sobreajuste de
grado 15 se refleja en sus coeficientes: para oscilar entre los puntos de ruido necesita
pesos enormes que se cancelan entre sí. La **regularización** penaliza justo eso —
agrega a la pérdida un costo por el tamaño de los pesos:

$$L_{\text{ridge}}(\mathbf{w}) = \lVert X\mathbf{w} - \mathbf{y} \rVert^2 + \alpha \lVert \mathbf{w} \rVert_2^2
\qquad \text{(Ridge, penalización L2)}$$

$$L_{\text{lasso}}(\mathbf{w}) = \lVert X\mathbf{w} - \mathbf{y} \rVert^2 + \alpha \lVert \mathbf{w} \rVert_1
\qquad \text{(Lasso, penalización L1)}$$

El **hiperparámetro** $\alpha \geq 0$ controla el trade-off: $\alpha = 0$ recupera
mínimos cuadrados; $\alpha \to \infty$ aplasta todos los pesos a cero. La diferencia
entre las dos penalizaciones es geométrica y tiene consecuencias prácticas:

- **Ridge** encoge todos los pesos suavemente hacia cero (para Ridge también existe
  solución cerrada: $\mathbf{w}^* = (X^\top X + \alpha I)^{-1} X^\top \mathbf{y}$ —
  la matriz se vuelve invertible *siempre*).
- **Lasso** lleva pesos exactamente a **cero**: hace selección de características
  automática. La esquina de la norma L1 es la responsable.

**Requisito práctico**: las penalizaciones comparan pesos entre sí, así que las
características deben estar en una escala común. De aquí en adelante, `StandardScaler`
(restar media, dividir entre desviación estándar) es parte estándar del pipeline.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

alphas_to_show = [0, 1e-4, 1]
degree = 15

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, alpha in zip(axes, alphas_to_show):
    model = make_pipeline(
        PolynomialFeatures(degree),
        StandardScaler(),
        Ridge(alpha=alpha) if alpha > 0 else LinearRegression(),
    )
    model.fit(x_train.reshape(-1, 1), y_train)
    ax.scatter(x_train, y_train, s=25, alpha=0.7)
    ax.plot(x_grid, true_function(x_grid), linewidth=1.5, color="C2", alpha=0.7)
    ax.plot(x_grid, model.predict(x_grid.reshape(-1, 1)), linewidth=2, color="C1")
    ax.set_ylim(-1.8, 1.8)
    ax.set_xlabel("x")
    ax.set_title(f"grado 15, α = {alpha}")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("y")
fig.suptitle("El mismo polinomio de grado 15: la regularización domestica la varianza", y=1.02)
fig.tight_layout()
plt.show()

## ¿Cómo elegir α? Validación cruzada

Tenemos que elegir $\alpha$ (y el grado, y en general los **hiperparámetros**: los que
el ajuste no aprende). La tentación es probar valores y quedarnos con el que dé mejor
error **de test**, pero eso invalida el test como medida de generalización: si lo usamos
para decidir, habríamos ajustado los hiperparámetros a él.

**Principio central: el test se toca una sola vez, al final de todo.**

La herramienta correcta es la **validación cruzada** (*k-fold CV*) dentro del train:

1. Divide el train en $k$ bloques (típicamente $k = 5$).
2. Entrena con $k-1$ bloques y valida con el restante; rota $k$ veces.
3. Promedia los $k$ errores de validación.

Cada punto se usa para validar exactamente una vez, y el promedio es mucho más estable
que un split único — especialmente con pocos datos. `GridSearchCV` automatiza el
recorrido sobre una rejilla de hiperparámetros con este protocolo.

In [ ]:
from sklearn.model_selection import GridSearchCV

pipeline = make_pipeline(
    PolynomialFeatures(),
    StandardScaler(),
    Ridge(),
)

param_grid = {
    "polynomialfeatures__degree": range(1, 16),
    "ridge__alpha": np.logspace(-6, 2, 9),
}

search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
)
search.fit(x_train.reshape(-1, 1), y_train)

best_rmse_cv = -search.best_score_
print(f"mejores hiperparámetros: {search.best_params_}")
print(f"RMSE de validación cruzada: {best_rmse_cv:.3f}")

# Ahora sí — y solo ahora — tocamos el test, una vez:
rmse_final = root_mean_squared_error(y_test, search.predict(x_test.reshape(-1, 1)))
print(f"RMSE final en test:         {rmse_final:.3f}   (σ del ruido = {noise_std})")

## De vuelta al mundo real: California Housing regularizado

Cerramos aplicando el protocolo completo — pipeline con escalado, búsqueda de $\alpha$
por validación cruzada, evaluación final en test — al dataset de la sesión pasada.
De paso, con las características **escaladas**, los coeficientes por fin son
comparables entre sí.

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split

housing = fetch_california_housing(data_home=DATA_DIR, as_frame=True)
X = housing.frame.drop(columns="MedHouseVal")
y = housing.frame["MedHouseVal"]

X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X, y, test_size=0.2, random_state=42
)

results = {}
for name, estimator in [("ridge", Ridge()), ("lasso", Lasso(max_iter=5000))]:
    pipe = make_pipeline(StandardScaler(), estimator)
    search_h = GridSearchCV(
        pipe,
        {f"{name}__alpha": np.logspace(-4, 2, 13)},
        cv=5,
        scoring="neg_root_mean_squared_error",
    )
    search_h.fit(X_train_h, y_train_h)
    rmse = root_mean_squared_error(y_test_h, search_h.predict(X_test_h))
    results[name] = search_h
    alpha_best = search_h.best_params_[f"{name}__alpha"]
    print(f"{name}:  mejor α = {alpha_best:.4f}   RMSE test = {rmse:.3f} [$100k]")

In [ ]:
coef_table = pd.DataFrame(
    {
        name: pd.Series(
            search.best_estimator_[-1].coef_, index=X.columns
        )
        for name, search in results.items()
    }
).sort_values("ridge")

fig, ax = plt.subplots(figsize=(8, 4.5))
y_pos = np.arange(len(coef_table))
bar_height = 0.38
ax.barh(y_pos + bar_height / 2, coef_table["ridge"], height=bar_height, label="ridge")
ax.barh(y_pos - bar_height / 2, coef_table["lasso"], height=bar_height, label="lasso")
ax.set_yticks(y_pos, coef_table.index)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("coeficiente [features escaladas — ahora sí comparables]")
ax.set_title("Ridge encoge; Lasso apaga (coeficientes exactamente en cero)")
ax.legend()
ax.grid(alpha=0.3, axis="x")
plt.show()

## Ejercicio

Trabaja en una copia de este notebook dentro de `mi-trabajo/`.

1. **Curva de validación con CV.** Repite la curva train/test del inicio, pero usando
   `cross_val_score` (5 folds) sobre el train en lugar del conjunto de test. ¿El grado
   óptimo coincide con el que encontró `GridSearchCV`?

2. **Lasso como detector de características irrelevantes.** Genera un dataset sintético
   con 10 características donde solo 3 afectan a $y$ (las otras 7 son ruido puro).
   Ajusta Lasso con varios valores de $\alpha$ y grafica los coeficientes.
   ¿Recupera cuáles características importan?

3. **Reto — el tamaño del dataset importa.** Repite el experimento de los polinomios con
   $n_{\text{train}} = 500$ en lugar de 25. ¿Sigue sobreajustando el grado 15?
   Conecta tu respuesta con el término de varianza de la descomposición.

## Apéndice A — Derivación de la descomposición sesgo–varianza

Sea $y = f(x) + \varepsilon$ con $\mathbb{E}[\varepsilon] = 0$,
$\operatorname{Var}(\varepsilon) = \sigma^2$, y sea $\hat{f}$ el modelo entrenado sobre
un dataset aleatorio $\mathcal{D}$ (la esperanza $\mathbb{E}$ es sobre datasets y ruido;
escribimos $\bar{f}(x) \equiv \mathbb{E}[\hat{f}(x)]$).

Sumamos y restamos $\bar{f}(x)$ dentro del error:

$$\mathbb{E}\big[ (y - \hat{f})^2 \big]
= \mathbb{E}\big[ (f + \varepsilon - \bar{f} + \bar{f} - \hat{f})^2 \big]$$

Expandimos el cuadrado del trinomio $(A + B + C)^2$ con $A = f - \bar{f}$ (constante),
$B = \varepsilon$, $C = \bar{f} - \hat{f}$:

$$= \underbrace{(f - \bar{f})^2}_{A^2}
+ \underbrace{\mathbb{E}[\varepsilon^2]}_{B^2}
+ \underbrace{\mathbb{E}\big[ (\bar{f} - \hat{f})^2 \big]}_{C^2}
+ 2\,\mathbb{E}[AB] + 2\,\mathbb{E}[AC] + 2\,\mathbb{E}[BC]$$

Los tres términos cruzados se anulan:

- $\mathbb{E}[AB] = (f - \bar{f})\,\mathbb{E}[\varepsilon] = 0$ — el ruido tiene media cero.
- $\mathbb{E}[AC] = (f - \bar{f})\,\mathbb{E}[\bar{f} - \hat{f}] = (f - \bar{f})(\bar{f} - \bar{f}) = 0$
  — por definición de $\bar{f}$.
- $\mathbb{E}[BC] = 0$ — el ruido del punto de evaluación es independiente del dataset
  de entrenamiento.

Queda exactamente

$$\mathbb{E}\big[ (y - \hat{f})^2 \big]
= \underbrace{(f - \bar{f})^2}_{\text{sesgo}^2}
+ \underbrace{\sigma^2}_{\text{ruido}}
+ \underbrace{\mathbb{E}\big[ (\hat{f} - \bar{f})^2 \big]}_{\text{varianza}} \qquad \blacksquare$$